In [1]:
import sys
sys.path.append('../../../../')

In [2]:
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem('transform_demo')

optimization_problem.add_variable('var_0')
optimization_problem.add_variable('var_1')
optimization_problem.add_variable('var_2')

[INFO 08-12 16:22:22] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


RangedParameter(name='var_2', parameter_type=float, lb=-inf, ub=inf)

In [3]:
def transform_fun(var_0, var_1):
    return var_0/var_1

optimization_problem.add_variable_dependency('var_2', ['var_0', 'var_1'], transform=transform_fun)

In [4]:
from examples.load_wash_elute.lwe_flow_rate import process
optimization_problem = OptimizationProblem('adsorption_rate_demo')
optimization_problem.add_evaluation_object(process)

In [5]:
optimization_problem.add_variable(
    name='adsorption_rate',
    parameter_path='flow_sheet.column.binding_model.adsorption_rate',
    lb=1e-3, ub=1e3,
    normalization='auto',
    indices=[1]  # modify only the protein (component index 1) parameter
)

optimization_problem.add_variable(
    name='desorption_rate',
    parameter_path='flow_sheet.column.binding_model.desorption_rate',
    lb=1e-3, ub=1e3,
    normalization='auto',
    indices=[1]
)

RangedParameter(name='desorption_rate', parameter_type=float, lb=0.001, ub=1000.0)

In [6]:
optimization_problem.add_variable(
    name='equilibrium_constant',
    evaluation_objects=None,
    lb=1e-4, ub=1e3,
    normalization='auto',
    indices=[1]
)

optimization_problem.add_variable(
    name='kinetic_constant',
    evaluation_objects=None,
    lb=1e-4, ub=1e3,
    normalization='auto',
    indices=[1]
)

RangedParameter(name='kinetic_constant', parameter_type=float, lb=0.0001, ub=1000.0)

In [7]:
optimization_problem.add_variable_dependency(
    dependent_variable="desorption_rate",
    independent_variables=["kinetic_constant", ],
    transform=lambda k_kin: 1 / k_kin
)

optimization_problem.add_variable_dependency(
    dependent_variable="adsorption_rate",
    independent_variables=["kinetic_constant", "equilibrium_constant"],
    transform=lambda k_kin, k_eq: k_eq / k_kin
)